# 01. Workflow와 Agent의 경계

오늘 과정 전체를 관통하는 질문은 "Agent를 몇 개 만들까?"가 아니라 **"어디에 판단을 맡길까?"**입니다. 에이전트의 개수는 그 질문에 답한 결과일 뿐입니다. 어디에 판단을 맡길지 경계가 정해지지 않은 상태에서 에이전트 수만 늘리면 오히려 실패 지점과 비용만 늘어납니다.

이 과정의 설계 지도는 다섯 단계로 이어집니다: **작업 목표 → 작업 그래프 → 판단 배치 → Context 분리 → 검증·운영.** 이번 노트북에서는 첫 번째 단계인 **작업 목표**에서 출발해 **판단 배치**까지 다룹니다. 동일한 고객 요청 한 건을 여러 방식으로 처리하면서, **다음 행동을 누가 결정하는지(결정권)**와 **실제 돈을 움직이는 쓰기 권한이 누구에게 있는지(실행 권한)**를 실행 기록으로 구분해 보겠습니다. 마지막에는 이후 노트북에서 모든 구조의 비교 기준점으로 삼을 **Single 기준선(Baseline)**을 남깁니다.

## 학습 목표

- 단순 LLM 호출, 고정 도구 사용, Agent Loop 세 가지 방식을 "관찰 뒤 다음 행동을 누가 고르는가"라는 기준으로 구분합니다.
- 동일한 고객 요청을 고정 경로(Workflow)와 동적 조사(Agent)로 처리한 실행 기록을 비교하고, 각 구조가 어떤 지점에서 어떻게 실패하는지 설명합니다.
- 하나의 그래프 안에서 Agent에게 맡길 노드와 결정적 코드로 남길 노드를 구분해 배치합니다.
- 모델의 제안(`Proposal`)과 시스템의 실행 권한(`check_refund`, `execute_approved`)을 분리하고, 프롬프트 지시만으로 막는 것과 코드 게이트로 막는 것의 차이를 실험으로 확인합니다.
- Single 기준선을 측정해 저장하고, 정보 부족(`insufficient`)과 실행 오류(`budget_exceeded`, `model_error`)를 구분합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 0 | 사례와 오늘의 질문 | 5 |
| 1 | 결정권: 세 가지 실행 방식 | 10 |
| 2 | 같은 요청, 두 구조 | 13 |
| 3 | Workflow 안의 Agent | 8 |
| 4 | 제안과 실행 권한 | 10 |
| 5 | Single 기준선 | 8 |
| 6 | 직접 해보기와 정리 | 6 |

## 호출 규모

이 노트북은 실제 모델로만 실행되며, API 키가 없으면 첫 번째 코드 셀에서 멈춥니다. 위에서 아래로 한 번 실행하면 **모델 호출 약 35~65회(실행 전 추정), EXA 호출 0회**가 발생하며 소요 시간은 약 2~5분입니다. `직접 해보기`의 유료 셀은 기본값(`RUN_EXERCISE = False`)으로 꺼져 있으며, 켜서 실행하면 모델 호출이 약 4~8회 추가됩니다. 유료 셀은 실행 직전에 `announce`로 호출 규모를 먼저 알리고, 모델을 새로 호출하지 않는 셀은 출력에 `"모델 호출 0회"`를 표시합니다.

In [ ]:
import inspect
import json
import time
from datetime import date, timedelta
from typing import TypedDict

from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph

from mas_lab.contracts import Proposal, Route, WorkerTask
from mas_lab.desk import Desk, load_billing, load_outage
from mas_lab.hybrid import ROLE_GOALS
from mas_lab.ledger import APPROVAL_THRESHOLD, REFUND_WINDOW_DAYS, PaymentLedger, check_refund, execute_approved
from mas_lab.runner import FIXED_BUDGET, build_retriever, run_case, run_grid
from mas_lab.trace import ModelMeter, timeline
from mas_lab.view import message_table, show_graph, show_table, write_design_log
from mas_lab.workers import allowlist_middleware, run_worker
from rag_lab.corpus import Scope, load_cases, load_corpus
from rag_lab.evidence import Budget, BudgetExceeded
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

# API 키가 설정되어 있는지 확인하고 모델과 검색기(BM25), 고객 범위를 초기화합니다.
require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
model = make_model()  # 모델 객체를 생성하며 아직 API 호출은 발생하지 않습니다.
scope, retriever = Scope("알파", date(2026, 9, 8)), build_retriever()
request = "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."  # 관통 사례 고객 요청 T-1001


def fresh_desk(for_scope=scope):
    """예산(budget)의 제한 시간은 Desk 생성 시점부터 측정되므로 실행마다 새 Desk를 생성합니다."""
    return Desk(for_scope, retriever, budget=Budget(**FIXED_BUDGET))


show_table([key_status()])
print(f"범위: 고객={scope.entity}, 기준일={scope.as_of} | 검색 조각 {len(retriever.chunks)}개 | 모델 호출 0회")

## 0절. 사례와 오늘의 질문 (5분)

오늘 과정 내내 동일한 고객 요청 하나를 여러 아키텍처로 처리합니다. 입력 사례를 고정해야 구조 사이의 차이를 사례 난이도 차이와 섞지 않고 정확히 비교할 수 있기 때문입니다. 고객(entity) `알파`가 기준일(as_of) `2026-09-08`에 보낸 아래 요청을 살펴보겠습니다(결제 내역, 가용률, 계약 원문은 모두 **수업용 가상 데이터**이며, 환불 처리는 노트북 메모리의 가상 원장에만 기록됩니다).

> 고객 요청 T-1001: "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."

짧은 한 문장이지만 실제로는 여러 단계의 작업이 결합되어 있습니다. 결제 내역을 조회해야 하고, 환불 규정을 찾아야 하며, 보상안을 안내하려면 월별 가용률과 SLA 계약 원문도 확인해야 합니다. 무엇보다 "취소해 주세요"는 단순 정보 조회가 아니라 **실제 돈을 움직이는 쓰기(write) 요청**입니다.

**먼저 예측하세요.** 이 요청을 끝까지 처리하려면 어떤 종류의 사실과 문서가 몇 개나 필요할까요? 또한 모델 호출은 최소 몇 번이면 충분할까요? 필요한 사실 목록과 예상 모델 호출 수를 먼저 적어 본 뒤 아래 코드 셀을 실행해 보세요(예상한 호출 수는 5절의 기준선 측정 표에서 다시 대조해 봅니다).

아래 코드에서는 고객 `알파`의 결제 내역, 월별 가용률, 그리고 적용 가능한 사내 원문 문서 목록을 불러와 확인합니다.

In [ ]:
# 고객 알파의 결제 내역과 월별 가용률·장애 기록을 조회합니다.
payments = load_billing(scope.entity)
show_table(payments, columns=["payment_id", "invoice_id", "approved_at", "amount", "base_fee_part", "usage_fee_part"])
show_table([{"월": month, "가용률": fact["availability_percent"], "사건": ", ".join(i["incident_id"] for i in fact["incidents"]) or "없음"}
            for month, fact in load_outage(scope.entity).items()])

# 사내 코퍼스 중 현재 고객 범위(scope)에 적용되는 원문 문서만 필터링하여 출력합니다.
corpus = load_corpus()
show_table([{"doc_id": d.doc_id, "제목": d.title, "버전": d.version, "담당자": d.owner} for d in corpus.values() if d.applies(scope)])
print(f"전체 문서 {len(corpus)}개 중 현재 고객 범위에 적용되는 문서만 출력했습니다(수업용 가상 데이터). 모델 호출 0회")

**관찰할 것.** 아직 모델은 한 번도 호출되지 않았습니다. 출력된 세 표를 다음 세 가지 범주로 나누어 살펴보세요.

| 종류 | 어디에 저장되어 있는가 | 이번 출력에서 확인할 열 |
|---|---|---|
| 이미 확정된 사실 | 사내 시스템의 결제·가용률 데이터 | `payment_id`, `invoice_id`, `amount`, `가용률` |
| 문서에 정의된 규칙 | 승인된 사내 원문 문서 | `doc_id`, `버전`, `담당자` |
| 아직 내려지지 않은 판단 | 어디에도 없음 (처리 과정에서 결정) | 어느 결제가 환불 대상인가, 환불과 보상을 어떻게 종합해 안내할 것인가 |

첫 번째 결제 표에서 동일한 청구서 번호(`invoice_id`)가 두 번 등장하는 행을 찾아보세요. 모델의 추론이나 판단이 필요한 곳은 세 번째 줄뿐입니다. 만약 첫 번째·두 번째 줄의 사실 조회까지 도구 없이 모델에게 맡기면 모델은 알 수 없는 값을 추측해 지어내게 됩니다.

## 1절. 결정권: 세 가지 실행 방식 (10분)

"LLM을 사용하면 모두 Agent다"라는 식의 넓은 정의는 시스템 설계에 도움이 되지 않습니다. 질문 의도를 분류하는 데만 LLM을 1회 호출하는 프로그램과, 도구 조회 결과를 보고 다음 행동을 스스로 결정하는 프로그램은 실패 양상과 비용 구조가 완전히 다르기 때문입니다. 따라서 이 과정에서는 도구의 유무가 아니라 **관찰 뒤 다음 행동을 누가 고르고, 언제 종료할지를 누가 결정하는가**를 기준으로 구분합니다.

| 실행 방식 | 관찰 뒤 다음 행동을 고르는 주체 | 종료 시점을 정하는 주체 |
|---|---|---|
| 단순 LLM 호출 | 선택할 다음 행동이 없습니다 (입력 1회에 출력 1회) | 1회 호출이 끝나면 즉시 종료됩니다 |
| 도구 사용 / LLM + 코드 분기 | **코드(`if`)**가 결정합니다 (모델은 분류 값만 반환) | **코드**가 종료를 결정합니다 |
| Agent Loop | **모델**이 결정합니다 (도구 결과를 보고 추가 호출 여부 판단) | **모델**이 결정합니다 (코드는 예산·재귀 상한만 강제) |

**먼저 예측하세요.** 아무런 조회 도구나 사실 데이터 없이 고객 요청 문장만 모델에게 그대로 보내면 어떻게 응답할까요? 금액이나 결제 번호를 임의로 지어낼지, 아니면 추가 정보를 요청할지 예측해 보세요. 또한 이어지는 네 가지 실행 방식 중 모델 호출 수가 가장 많을 방식과, 모델이 스스로 다음 행동을 선택하는 방식이 어느 것인지 미리 적어 보세요.

아래 코드에서는 도구 없이 고객 요청 문장만 LLM에 전달해 응답을 확인합니다.

In [ ]:
# 1. 도구와 사실 데이터 없이 질문만 모델에 전달하여 응답을 확인합니다.
announce("LLM 호출: 사실을 주지 않고 묻기", model_calls=1)
meter_llm = ModelMeter()
reply_llm = model.invoke(request, config={"callbacks": [meter_llm]})
print(reply_llm.text)
print("\ntool_calls:", reply_llm.tool_calls)
print("토큰 사용량:", reply_llm.usage_metadata)

**관찰할 것.** 모델의 응답 본문에 `P-`로 시작하는 결제 번호나 구체적인 금액이 포함되어 있는지 확인해 보세요. 만약 구체적인 번호나 금액이 있다면 모델이 근거 없이 지어낸(환각) 것입니다. 반대로 "결제 내역을 알려 주세요"나 "고객센터로 문의해 주세요"처럼 답했다면 거짓 사실을 지어내지는 않았지만 **고객의 요청은 전혀 처리되지 않은 상태**입니다. 또한 환불 소요일이나 일반적인 절차 안내가 포함되어 있다면 이는 사내 원문 규정이 아니라 모델의 사전 학습 지식일 뿐입니다. 도구를 제공하지 않았으므로 `tool_calls`는 빈 리스트(`[]`)로 출력됩니다.

아래 코드에서는 결제 조회 도구를 사용하되, **코드가 인자를 직접 지정해 호출**하고 조회 결과에 따른 다음 행동도 코드의 `if` 조건문으로 결정하는 방식을 확인합니다.

In [ ]:
# 2. 코드가 직접 도구를 호출하고 if 조건문으로 다음 행동을 결정합니다(모델 호출 0회).
desk_tool = fresh_desk()
lookup_payments = desk_tool.tools("billing")[0]
found = lookup_payments.invoke({"invoice_id": "INV-2026-08"})
show_table(found["payments"], columns=["payment_id", "invoice_id", "approved_at", "amount", "fact_ref"])

next_step = "환불 대상 후보로 표시" if len(found["payments"]) >= 2 else "중복 아님으로 종료"
print(f"도구: {lookup_payments.name} | 도구 호출 1회 | 다음 행동(코드의 if가 결정): {next_step} | 모델 호출 0회")

이번에는 LLM 모델을 사용하되, 모델의 역할을 정해진 스키마(`Route`) 값 하나를 출력하는 것으로 제한해 보겠습니다. 아래 코드에서는 모델이 질문의 경로(`path`)와 필요한 조사 영역(`roles`)만 분류하고, 실제로 다음에 무엇을 실행할지는 Python 함수 `next_by_code`의 `if` 조건문이 결정합니다. **LLM을 사용하지만 Agent가 아닌 전형적인 Workflow 분기** 예시입니다. 정상 업무 요청과 창구 범위 밖 질문("사내 식당 메뉴를 알려 주세요.") 두 가지를 넣어 결과를 비교합니다.

In [ ]:
# 3. LLM으로 구조화된 분류(Route)만 생성하고, 다음 행동은 코드(next_by_code)가 결정합니다.
ROUTE_PROMPT = ("고객 요청의 경로를 고르세요. fast: 규정 한 곳이나 결제 한 건만 보면 되는 질문. slow: 결제, 규정, 장애 중 둘 이상이 필요한 질문. "
                "out_of_scope: 결제, 계약, 장애 업무가 아닌 질문. roles에는 필요한 조사를 고르세요. billing: 결제 사실. policy: 환불 규정과 SLA 문서. "
                "outage: 가용률.\n질문: {question}")


def next_by_code(route: Route) -> str:
    """모델이 분류한 Route 값을 받아 코드의 조건문으로 다음 단계를 결정합니다."""
    if route.path == "out_of_scope":
        return "담당자에게 전환"
    return "한 곳만 조회" if route.path == "fast" else "조회 목록을 차례로 실행: " + ", ".join(route.roles)


announce("LLM + 코드 분기: 의도만 분류", model_calls=2)
meter_route, router = ModelMeter(), model.with_structured_output(Route, method="json_schema")
routes = []
for question in [request, "사내 식당 메뉴를 알려 주세요."]:
    route = router.invoke(ROUTE_PROMPT.format(question=question), config={"callbacks": [meter_route]})
    routes.append({"요청": question, "path": route.path, "roles": route.roles, "다음 행동(if가 고름)": next_by_code(route)})
show_table(routes)

마지막은 **Agent Loop**입니다. 결제 조회 도구(`lookup_payments`) 하나만 제공하고, 어떤 인자로 호출할지, 몇 번 호출할지, 언제 최종 답변을 작성할지는 코드가 정하지 않고 모델에게 맡깁니다(`create_deep_agent`가 기본으로 붙이는 파일 시스템 도구는 `allowlist_middleware`로 차단해 `lookup_payments`만 노출했습니다). 메시지 표에서 `종류`가 `AI`이고 `인자` 열이 채워진 행이 바로 모델이 스스로 다음 행동을 선택한 지점입니다.

아래 코드에서는 Agent Loop를 실행한 뒤 메시지 기록과 네 가지 실행 방식의 비교표를 출력합니다.

In [ ]:
# 4. Agent Loop: 모델이 스스로 도구 호출 여부와 종료 시점을 결정합니다.
desk_agent = fresh_desk()
agent_loop = create_deep_agent(
    model=model, tools=desk_agent.tools("billing"), name="agent-loop",
    system_prompt=f"고객 요청을 돕는 Agent입니다. 고객={scope.entity}, 기준일={scope.as_of}. 도구로 확인한 사실만 답하세요.",
    middleware=allowlist_middleware(desk_agent.budget, frozenset({"lookup_payments"})))

announce("Agent Loop: 결제 조회 도구 하나", model_calls=4)
meter_agent = ModelMeter()
state_agent = agent_loop.invoke({"messages": [{"role": "user", "content": request}]},
                                config={"callbacks": [meter_agent], "recursion_limit": 40})
show_table(message_table(state_agent["messages"]))
print(state_agent["messages"][-1].text)

In [ ]:
# 앞선 네 가지 실행 방식의 모델 호출 수, 도구 호출 수, 결정 주체를 한눈에 비교합니다(모델 호출 0회).
comparison = [
    {"실행": "LLM 호출", "모델 호출": meter_llm.totals()["model_calls"], "도구 호출": len(reply_llm.tool_calls),
     "다음 행동을 고른 주체": "없음", "종료를 정한 주체": "호출 한 번으로 끝"},
    {"실행": "도구 사용", "모델 호출": 0, "도구 호출": 1, "다음 행동을 고른 주체": "코드", "종료를 정한 주체": "코드"},
    {"실행": "LLM + 코드 분기", "모델 호출": meter_route.totals()["model_calls"], "도구 호출": 0,
     "다음 행동을 고른 주체": "코드(if)", "종료를 정한 주체": "코드"},
    {"실행": "Agent Loop", "모델 호출": meter_agent.totals()["model_calls"], "도구 호출": desk_agent.budget.counts.get("tool", 0),
     "다음 행동을 고른 주체": "모델", "종료를 정한 주체": "모델(코드는 상한만)"},
]
show_table(comparison)
print("이 셀 자체는 모델 호출 0회입니다.")

**관찰할 것.** 비교표에서 `다음 행동을 고른 주체`와 `종료를 정한 주체`는 아키텍처가 결정하는 고정 속성이므로 실행할 때마다 바뀌지 않습니다. 반면 `모델 호출`과 `도구 호출`은 실행 시점에 따라 달라질 수 있으며, 실행 전에 호출 횟수를 미리 확정할 수 없는 유일한 방식은 **Agent Loop**입니다.

- **Route 분류 표**: 두 번째 질문("사내 식당 메뉴를 알려 주세요.")이 `out_of_scope`로 분류되었다면 코드의 `if`가 담당자 전환을 선택한 것입니다. 설령 모델이 분류를 틀리더라도 **코드에 정의된 분기 목록 밖의 행동은 절대 실행되지 않습니다.** 또한 T-1001 요청의 `roles`에 `outage`(가용률)까지 포함되었는지 확인해 보세요.
- **Agent Loop 메시지 표**: 모델이 `lookup_payments`를 어떤 인자로 몇 번 호출했는지 살펴봅니다. 인자 없이 전체 결제를 조회했든, 특정 청구서를 지정했든 그 선택은 코드가 아니라 모델이 내린 것입니다.
- **Agent의 최종 답변**: 현재 에이전트에게는 결제 조회 도구만 있고 규정·가용률 조회 도구는 없습니다. 보상안이나 환불 규정에 대해 모른다고 답했다면 도구의 범위가 답변의 경계를 올바르게 제한한 것이고, 확인되지 않은 규정을 언급했다면 근거 없는 추측이 섞인 것입니다.

이 세 가지 방식은 우열 관계가 아니라 업무 특성에 따른 선택지입니다. 단순 의도 분류 하나를 Agent Loop로 만들면 호출 횟수와 출력 스키마의 예측 가능성을 잃고 비용만 늘어나게 됩니다.

## 2절. 같은 요청, 두 구조 (13분)

Workflow와 Agent는 **서로 다른 지점에서 다른 방식으로 실패**합니다. 이번 절에서는 키워드 규칙으로 분기하는 고정 경로(`fixed_route`)와 모든 조회 도구를 가진 단일 에이전트(`Single`)에 동일한 요청들을 넣어 두 구조의 실패 양상을 나란히 비교해 보겠습니다.

**먼저 예측하세요.** 키워드 규칙 기반 고정 경로(`fixed_route`)에 다음 네 가지 요청을 넣는다고 해 봅시다: (1) 같은 청구서를 두 번 결제하면? (2) 크레딧 신청 기한은? (3) T-1001 원문 (4) "지난달 장애 때문에 환불도 되나요?" 각 요청이 어느 분기로 이동할지, 그리고 그 분기의 응답이 질문 전체를 해결할 수 있을지 예측해 보세요.

아래 코드에서는 `fixed_route` 함수의 구현을 확인하고, 복합 요청과 우회 표현을 넣었을 때 어떤 한계가 나타나는지 재현합니다.

In [ ]:
def fixed_route(request: str) -> dict:
    """키워드 포함 여부로 경로를 분기합니다. 먼저 매칭된 if 조건 하나가 전체 경로를 결정합니다."""
    desk = fresh_desk()
    if "결제" in request:
        rows = desk.tools("billing")[0].invoke({})["payments"]
        later = [p["payment_id"] for i, p in enumerate(rows) if p["invoice_id"] in {q["invoice_id"] for q in rows[:i]}]
        return {"경로": "billing", "응답": f"중복 결제 후보: {', '.join(later)}", "도구": ["lookup_payments"]}
    if "기한" in request:
        hit = desk.context.search(request, limit=1)["evidence"][0]
        return {"경로": "deadline", "응답": f"{hit['doc_id']}: {' '.join(hit['quote'].splitlines())[:70]}", "도구": ["search_knowledge"]}
    if "장애" in request:
        fact = desk.tools("outage")[0].invoke({"month": "2026-08"})
        return {"경로": "outage", "응답": f"2026-08 가용률 {fact['availability_percent']}%", "도구": ["lookup_availability"]}
    return {"경로": "전환", "응답": "담당자에게 전환합니다.", "도구": []}


# 고정 경로 함수의 소스와 키워드 분기 우선순위 표를 출력합니다.
print(inspect.getsource(fixed_route))
show_table([{"먼저 볼 키워드": "결제", "경로": "billing"}, {"먼저 볼 키워드": "기한", "경로": "deadline"},
            {"먼저 볼 키워드": "장애", "경로": "outage"}, {"먼저 볼 키워드": "그 밖", "경로": "전환"}])
print("모델 호출 0회")

In [ ]:
# 실패 재현: 키워드 고정 경로에 복합 요청과 다른 표현의 요청을 넣어 누락되는 의도를 확인합니다
REQUESTS = ["같은 청구서를 두 번 결제하면 어떻게 처리되나요?", "서비스 크레딧 신청 기한은 언제까지인가요?",
            request, "지난달 장애 때문에 환불도 되나요?"]
COVERS = {"billing": {"결제", "취소", "환불"}, "deadline": {"기한", "크레딧", "신청"}, "outage": {"장애"}, "전환": set()}
SIGNALS = ["결제", "취소", "환불", "보상", "기한", "크레딧", "신청", "장애"]


def run_fixed(requests: list[str]) -> list[dict]:
    """고정 경로를 실행하고, 선택된 경로가 처리하지 못한 핵심 단어(경로가 보지 않은 단어)를 기록합니다."""
    rows = []
    for text in requests:
        started = time.perf_counter()
        result = fixed_route(text)
        missed = [word for word in SIGNALS if word in text and word not in COVERS[result["경로"]]]
        rows.append({"요청": text, "경로": result["경로"], "응답": result["응답"], "경로가 보지 않은 단어": ", ".join(missed) or "없음",
                     "도구": result["도구"], "시간(초)": round(time.perf_counter() - started, 4)})
    return rows


fixed_rows = run_fixed(REQUESTS)
show_table(fixed_rows, columns=["요청", "경로", "응답", "경로가 보지 않은 단어", "시간(초)"])
print("같은 의미를 다른 단어로 표현하거나 두 키워드가 함께 들어오면 어떻게 될까요?")
show_table(run_fixed(["돈이 두 번 빠져나갔어요.", "결제 기한을 넘기면 어떻게 되나요?"]), columns=["요청", "경로", "응답", "경로가 보지 않은 단어"])
print("모델 호출 0회")

**관찰할 것.** 표의 `경로` 열과 `경로가 보지 않은 단어` 열을 나란히 살펴봅니다. 고정 경로의 실패는 오류(Exception)를 일으키지 않고 **조용히 요청의 일부를 누락**시킨다는 점이 핵심입니다.

- **T-1001 복합 요청 행**: `"결제"` 키워드에 먼저 걸려 `billing` 경로로만 이동하면서 `경로가 보지 않은 단어`에 `보상`이 남았습니다. 고객이 요청한 보상안 확인은 아무런 경고 없이 버려졌습니다.
- **"지난달 장애 때문에 환불도 되나요?" 행**: `"장애"`와 `"환불"`이 함께 들어 있지만 `outage` 경로로만 빠져 가용률 숫자만 반환하고 정작 고객이 물은 환불 가능 여부에는 답하지 못했습니다.
- **기한 문의 행**: 검색 상위 1개 조각만 읽습니다. 고객 `알파`에게는 신청 기한을 다음 달 20일로 연장한 개별 약정(`sla-a-amendment`)이 있는데, 기본 약정(`sla-a-v2`, 15일) 조각 하나만 읽고 멈추면 경로는 맞았더라도 오답을 안내하게 됩니다.
- **우회 표현 표**: `"돈이 두 번 빠져나갔어요."`는 중복 결제 문의지만 `"결제"`라는 단어가 없어 담당자 전환으로 빠지고, `"결제 기한을 넘기면..."`은 앞선 `"결제"` 조건에 가로채여 기한 안내로 가지 못합니다.

물론 고정 경로의 장점도 분명합니다. 실행 시간이 수 밀리초(ms) 수준으로 빠르고, 실패 원인이 코드의 어느 줄인지 즉시 찾을 수 있습니다. 다만 예외 조합이 늘어날수록 규칙으로 모든 경우를 덮기 어려워집니다.

이제 조건을 하나만 바꾸어, 경로 선택을 코드가 아니라 모델이 수행하는 **Single 에이전트**에 세 번째·네 번째 요청을 넣어 비교해 보겠습니다.

**먼저 예측하세요.** Single 에이전트는 네 번째 요청("지난달 장애 때문에 환불도 되나요?")을 받았을 때 결제 내역, 가용률, 규정 문서 중 무엇을 어떤 순서로 조회할까요?

아래 세 코드 셀에서는 Single 에이전트를 실행하고 고정 경로와의 비용·도구 호출 순서 차이를 비교합니다.

In [ ]:
# 세 번째(T-1001)와 네 번째 요청을 전체 조회 도구를 가진 Single 구조로 실행합니다.
announce("Single: 복합 요청과 네 번째 요청", model_calls=12)
adhoc_cases = [{"id": f"adhoc-{index}", "type": "adhoc", "entity": scope.entity, "as_of": scope.as_of.isoformat(), "question": text}
               for index, text in enumerate(REQUESTS[2:], start=3)]
rows_single = [run_case("single", case, model=model) for case in adhoc_cases]
show_table(rows_single, columns=["case_id", "status", "outcome", "model_calls", "tool_calls", "elapsed_seconds", "errors"])
for row in rows_single:
    print(f"\n[{row['case_id']}] {row['answer']}")

In [ ]:
# 동일한 두 요청에 대해 고정 경로와 Single 에이전트의 비용·시간·도구 사용량을 비교합니다(모델 호출 0회).
two_structures = []
for fixed, single in zip(fixed_rows[2:], rows_single, strict=True):
    two_structures.append({"요청": fixed["요청"], "구조": "고정 경로", "경로 또는 outcome": fixed["경로"], "모델 호출": 0, "입력 토큰": 0,
                           "시간(초)": fixed["시간(초)"], "도구 호출": len(fixed["도구"]), "도구 종류": len(set(fixed["도구"]))})
    two_structures.append({"요청": fixed["요청"], "구조": "Single", "경로 또는 outcome": f"{single['status']} / {single['outcome']}",
                           "모델 호출": single["model_calls"], "입력 토큰": single["input_tokens"], "시간(초)": single["elapsed_seconds"],
                           "도구 호출": len(single["tools_called"]), "도구 종류": len(set(single["tools_called"]))})
show_table(two_structures)
for single in rows_single:
    print(f"Single {single['case_id']} 사용한 도구:", ", ".join(sorted(set(single["tools_called"]))))
print("이 셀 자체는 모델 호출 0회입니다. 토큰이 None이면 usage 누락이며 0이 아닙니다.")

In [ ]:
def tool_steps(row: dict) -> list[str]:
    """모델이 도구를 선택한 AI 메시지마다 호출한 도구 이름을 순서대로 추출합니다(쉼표는 한 턴 내 동시 호출)."""
    return [m["도구"] for m in row["messages"] if m["종류"] == "AI" and m["인자"]]


# 고정 경로와 Single 에이전트의 도구 선택 횟수 및 순서를 비교합니다(모델 호출 0회).
order_rows = [{"요청": f["요청"], "구조": "고정 경로", "도구를 고른 메시지": len(f["도구"]), "실행마다 달라지는가": "아니오. 코드에 고정되어 있습니다"} for f in fixed_rows[2:]]
order_rows += [{"요청": REQUESTS[2 + i], "구조": "Single", "도구를 고른 메시지": len(tool_steps(row)),
                "실행마다 달라지는가": "예. 이번 1회 관찰의 순서입니다"} for i, row in enumerate(rows_single)]
show_table(order_rows)
for fixed in fixed_rows[2:]:
    print("고정 경로 |", fixed["요청"][:16], "|", " > ".join(fixed["도구"]))
for i, row in enumerate(rows_single):
    print("Single    |", REQUESTS[2 + i][:16], "|", " > ".join(tool_steps(row)) or "(도구 없음)")
print("모델 호출 0회")

**관찰할 것.** 동일한 요청에 대한 고정 경로와 Single 에이전트 두 행을 짝지어 비교해 봅니다.

- **`경로 또는 outcome`**: Single의 결과가 `ok / needs_approval`이면 모델이 중복 결제 환불 제안(`Proposal`)을 제출했고 코드 게이트가 승인 필요 건으로 판정한 것입니다. `ok / answered`이면 환불 제안 없이 조회한 근거로 답변한 것입니다. 만약 `validation_failed`가 나왔다면 답변은 작성했으나 근거 번호(`evidence_refs`)나 금액 출처 검증에서 걸러진 경우입니다.
- **`사용한 도구` 목록**: Single 에이전트가 네 번째 요청에서 가용률 조회뿐 아니라 규정 검색(`search_knowledge`, `read_document`)까지 함께 수행했는지 확인해 보세요. 고정 경로가 놓쳤던 복합 질문을 처리한 대신, 그 대가로 `모델 호출` 횟수와 `입력 토큰`, `시간(초)`를 지불한 것입니다.
- **도구 호출 순서**: 출력의 `>` 기호는 다음 턴에서 모델이 이어서 도구를 선택한 순서이고, 쉼표(`,`)는 한 메시지 안에서 여러 도구를 동시에 호출한 것입니다.

두 구조는 실패를 디버깅하는 방식도 다릅니다. **고정 경로의 실패는 코드의 조건문 한 줄에서 바로 드러나지만, Agent의 실패는 메시지 궤적 전체를 읽어야 원인을 파악할 수 있습니다.** 그래서 실무에서는 둘 중 하나만 고집하기보다 **하나의 그래프 안에 Workflow와 Agent를 적재적소에 섞어 배치**합니다.

## 3절. Workflow 안의 Agent (8분)

Single 에이전트는 고객 등록 여부 확인이나 단순 결제 내역 조회처럼 결정적 코드로 충분한 작업까지 매번 LLM 호출로 처리합니다. 단계가 길어질수록 이전 메시지 전체가 입력 토큰으로 반복 전달되어 비용과 지연이 누적됩니다. 이제 질문을 바꾸어 **요청 처리의 각 단계마다 가장 저렴하고 충분한 수단이 무엇인지** 나누어 배치해 보겠습니다.

| 노드 분류 | 의미 | 적합한 상황 | 이번 그래프의 노드 |
|---|---|---|---|
| `CODE` | 일반 Python 함수 | 입력이 정해지면 결과도 결정적으로 정해질 때 | `verify`: 등록된 고객 범위인지 확인 |
| `MODEL` | 단일 LLM 호출 | 자연어 해석이 필요하지만 반복 루프는 불필요할 때 | `interpret`: 질문 의도 및 필요 조사 분류 |
| `API` | 시스템 데이터 조회 | 필요한 사실 데이터가 이미 사내 시스템에 있을 때 | `fetch`: 결제 내역 및 월별 가용률 조회 |
| `AGENT` | Agent Loop | 검색 결과를 보며 다음 검색어나 문서를 골라야 할 때 | `investigate`: 환불 규정 및 개별 약정 조사 |
| `RULE` | 결정적 규칙 코드 | 회사 정책을 조건문으로 명확히 판정해야 할 때 | `rule`: 중복 결제 환불 판정 (`check_refund`) |

아래 두 코드 셀에서는 이 다섯 단계를 LangGraph의 `StateGraph`로 조립합니다. `verify` 노드 뒤에는 조건부 간선을 두어, 등록되지 않은 고객이면 유료 모델을 호출하기 전에 즉시 종료(`END`)하도록 구성합니다.

**먼저 예측하세요.** 다섯 노드 중 실행 시간을 가장 많이 쓰는 노드는 어디일까요? 또한 `interpret` 노드가 `roles`에서 `billing`이나 `outage`를 누락하면 뒤쪽 노드의 출력이 어떻게 달라질지 예측해 보세요.

In [ ]:
class CaseState(TypedDict, total=False):
    question: str
    stop: str
    route: dict
    facts: dict
    findings: dict
    decisions: list


def code_nodes(desk: Desk) -> dict:
    """CODE, API, RULE 역할을 맡는 결정적 노드 함수들을 생성합니다(모델 호출 0회)."""
    def verify(state: CaseState) -> dict:
        return {"stop": "" if desk.payments() else f"등록되지 않은 고객입니다: {desk.scope.entity}"}

    def fetch(state: CaseState) -> dict:
        roles, facts = state["route"]["roles"], {}
        if "billing" in roles:
            facts["payments"] = desk.tools("billing")[0].invoke({})["payments"]
        if "outage" in roles:
            month = (desk.scope.as_of.replace(day=1) - timedelta(days=1)).strftime("%Y-%m")
            facts["availability"] = desk.tools("outage")[0].invoke({"month": month})
        return {"facts": facts}

    def rule(state: CaseState) -> dict:
        if "payments" not in state["facts"]:
            return {"decisions": [{"결제": "-", "결정": "판단 불가", "이유": "결제 사실을 조회하지 않았습니다."}]}
        rows, seen, decisions = sorted(state["facts"]["payments"], key=lambda p: p["approved_at"]), set(), []
        for payment in rows:
            if payment["invoice_id"] in seen:
                proposal = Proposal(action="refund", payment_id=payment["payment_id"], amount=payment["amount"])
                verdict = check_refund(proposal, rows, desk.scope.as_of)
                decisions.append({"결제": payment["payment_id"], "결정": verdict.decision, "이유": verdict.reasons[0]})
            seen.add(payment["invoice_id"])
        return {"decisions": decisions}

    return {"verify": verify, "fetch": fetch, "rule": rule}


print("코드 노드 목록:", list(code_nodes(fresh_desk())), "| 상태 키:", list(CaseState.__annotations__), "| 모델 호출 0회")

In [ ]:
NODE_KIND = {"verify": "CODE", "interpret": "MODEL", "fetch": "API", "investigate": "AGENT", "rule": "RULE"}


def build_mixed(desk: Desk, meter: ModelMeter, *, interpret=None):
    """verify → interpret → fetch → investigate → rule 순서의 혼합 그래프를 조립합니다."""
    def interpret_with_model(state: CaseState) -> dict:
        route = model.with_structured_output(Route, method="json_schema").invoke(ROUTE_PROMPT.format(question=state["question"]))
        return {"route": route.model_dump()}

    def investigate(state: CaseState) -> dict:
        if "policy" not in state["route"]["roles"]:
            return {"findings": {"status": "조사하지 않음"}}
        task = WorkerTask(task_id="policy", goal=ROLE_GOALS["policy"], facts=state["facts"])
        return {"findings": run_worker(task, desk, model).model_dump()}

    def measured(name: str, fn):
        """각 노드의 실행 시간과 모델 호출 수를 Trace에 기록하는 래퍼입니다."""
        def node(state: CaseState) -> dict:
            before = len(meter.calls)
            with desk.trace.span(name, "node"):
                update = fn(state)
            desk.trace.event("node", name, model_calls=len(meter.calls) - before)
            return update
        return node

    fns = {**code_nodes(desk), "interpret": interpret or interpret_with_model, "investigate": investigate}
    graph = StateGraph(CaseState)
    for name in NODE_KIND:
        graph.add_node(name, measured(name, fns[name]))
    graph.add_edge(START, "verify")
    graph.add_conditional_edges("verify", lambda state: END if state["stop"] else "interpret", ["interpret", END])
    for left, right in [("interpret", "fetch"), ("fetch", "investigate"), ("investigate", "rule"), ("rule", END)]:
        graph.add_edge(left, right)
    return graph.compile()


# 혼합 그래프의 구조를 시각화합니다(아직 실행하지 않았으므로 모델 호출 0회).
show_graph(build_mixed(fresh_desk(), ModelMeter()))
print("그래프 정의만 완료했습니다. 모델 호출 0회")

In [ ]:
# 혼합 Workflow 그래프에 관통 사례 요청(T-1001)을 넣어 실행합니다.
announce("혼합 Workflow: 모델은 interpret와 investigate에서만", model_calls=6)
desk_mixed, meter_mixed = fresh_desk(), ModelMeter()
try:
    state_mixed = build_mixed(desk_mixed, meter_mixed).invoke({"question": request}, config={"callbacks": [meter_mixed]})
except BudgetExceeded as error:  # 예산 초과 발생 시에도 예외로 중단하지 않고 실패 상태로 기록합니다.
    state_mixed = {"route": {}, "facts": {}, "findings": {"status": "budget_exceeded", "error": str(error)}, "decisions": []}

print("분류 경로(route):", state_mixed["route"])
print("조회한 사실 키:", list(state_mixed["facts"]))
findings = state_mixed["findings"]
print("조사 상태:", findings.get("status"), "| 근거 수:", len(findings.get("evidence_refs", [])), "| 불확실성:", findings.get("uncertainty", []))
for fact in findings.get("facts", [])[:6]:
    print(" -", fact[:200])
show_table(state_mixed["decisions"])

In [ ]:
def node_rows(desk: Desk) -> list[dict]:
    """각 노드의 종류, 소요 시간(초), 모델 호출 수를 표 행으로 정리합니다."""
    spans = {s["actor"]: s for s in desk.trace.spans if s["kind"] == "node"}
    return [{"노드": e["actor"], "종류": NODE_KIND[e["actor"]], "시간(초)": round(spans[e["actor"]]["end"] - spans[e["actor"]]["start"], 3),
             "모델 호출": e["model_calls"]} for e in desk.trace.events if e["kind"] == "node"]


# 노드별 소요 시간·모델 호출 수 표와 타임라인 막대를 출력합니다.
show_table(node_rows(desk_mixed))
print(timeline([s for s in desk_mixed.trace.spans if s["kind"] == "node"]))
print("\n모델 호출 및 토큰 합계:", meter_mixed.totals())

In [ ]:
# 실패 재현: 등록되지 않은 고객(감마)으로 실행했을 때 verify 노드가 모델 호출 전에 즉시 중단하는지 확인합니다
desk_unknown, meter_unknown = fresh_desk(Scope("감마", date(2026, 9, 8))), ModelMeter()
state_unknown = build_mixed(desk_unknown, meter_unknown).invoke({"question": request}, config={"callbacks": [meter_unknown]})
print("중단 사유(stop):", state_unknown["stop"])
show_table(node_rows(desk_unknown))
print("모델 호출", meter_unknown.totals()["model_calls"], "회")

**관찰할 것.** 노드 실행 표의 `시간(초)`와 `모델 호출` 열을 함께 비교해 보세요.

- **노드별 비용과 시간 집중도**: 전체 소요 시간과 모델 호출이 `investigate`(AGENT) 노드에 집중되어 있고, `verify`(CODE), `fetch`(API), `rule`(RULE) 세 노드의 `모델 호출`은 정확히 `0`입니다.
- **앞 노드의 분류가 뒤 노드에 미치는 영향**: `route`의 `roles`에 `billing`이 포함되었기 때문에 `fetch`가 결제 사실을 조회했고, `rule` 노드가 중복 결제인 `P-9002`와 `P-7002`에 대해 각각 `needs_approval`(승인 기준 초과)과 `human_review`(90일 경과) 판정을 코드로 정확히 내렸습니다.
- **사전 차단(`verify`) 효과**: 마지막 실패 재현 셀(미등록 고객 `감마`)의 노드 표에는 `verify` 한 줄만 기록되어 있으며 모델 호출은 0회입니다. 고객 범위 확인을 모델에게 맡기지 않고 코드 노드로 앞단에 배치했기 때문에 불필요한 API 비용을 쓰기 전에 즉시 차단할 수 있습니다.

여기서 한 가지 질문을 더 던져 볼 수 있습니다: **`interpret` 노드도 규칙 코드로 대체할 수 있을까요?** `직접 해보기 4`에서 `interpret`를 규칙 함수로 바꾸었을 때 비용 절감과 표현 누락 위험이 어떻게 맞바뀌는지 직접 확인해 봅니다.

## 4절. 제안과 실행 권한 (10분)

지금까지는 "다음 행동을 누가 고르는가(결정권)"를 살펴보았습니다. 이제 두 번째 핵심 축인 **"고른 행동을 실제로 실행할 권한은 누구에게 있는가(실행 권한)"**를 다룹니다. 조회 도구를 잘못 호출하면 토큰 비용만 들지만, 환불 실행 도구를 잘못 호출하면 실제 돈이 움직입니다.

결제 번호 `P-8101`은 청구서 하나에 1건만 결제된 정상 결제입니다. 그런데 고객이 "P-8101도 중복 결제 같으니 환불해 주세요"라고 요청했다고 가정해 봅시다. 아래 첫 번째 실험에서는 일부러 위험한 구조를 만들어 봅니다. 모델에게 즉시 원장에 환불을 기록하는 `refund_now` 도구를 직접 부여하고, 시스템 프롬프트에 `"환불 규정을 지키세요."`라고만 지시합니다.

**먼저 예측하세요.** 모델은 `P-8101`을 먼저 조회한 뒤 거절할까요, 아니면 바로 환불 도구를 호출할까요? 더 중요한 질문은 이것입니다: **만약 이번 1회 실행에서 모델이 환불을 거절했다면, 이 구조를 프로덕션에 배포해도 안전하다고 말할 수 있을까요?**

아래 두 코드 셀에서는 환불 도구를 직접 가진 에이전트를 실행하고 실제 원장(`ledger.refunds`) 기록 여부를 확인합니다.

In [ ]:
# 실패 재현: 환불 실행 도구(refund_now)를 모델에게 직접 부여하고 프롬프트 지시만으로 통제를 시도합니다
REFUND_REQUEST = "P-8101도 중복 같으니 환불해 주세요."
WEAK_PROMPT = f"고객 요청을 처리하는 Agent입니다. 고객={scope.entity}, 기준일={scope.as_of}. 환불 규정을 지키세요."


def make_refund_agent(desk: Desk, system_prompt: str):
    """모델이 직접 원장에 환불을 기록하는 refund_now 도구를 호출할 수 있는 에이전트를 생성합니다."""
    @tool
    def refund_now(payment_id: str, amount: int) -> dict:
        """결제를 즉시 환불합니다. 호출하면 바로 원장에 기록됩니다."""
        return desk.ledger.refund(payment_id, amount)

    return create_deep_agent(model=model, tools=[*desk.tools("billing"), refund_now], system_prompt=system_prompt, name="refund-agent",
                             middleware=allowlist_middleware(desk.budget, frozenset({"lookup_payments", "refund_now"})))


announce("환불 도구를 직접 준 Agent", model_calls=4)
desk_refund = fresh_desk()
state_refund = make_refund_agent(desk_refund, WEAK_PROMPT).invoke({"messages": [{"role": "user", "content": REFUND_REQUEST}]},
                                                                  config={"recursion_limit": 40})
print(state_refund["messages"][-1].text)

In [ ]:
# 에이전트의 도구 호출 내역과 실제 결제 원장(ledger.refunds)의 변경 여부를 확인합니다(모델 호출 0회).
show_table(message_table(state_refund["messages"]))
called = [t["name"] for m in state_refund["messages"] for t in (getattr(m, "tool_calls", None) or [])]
show_table(desk_refund.ledger.refunds, columns=["refund_id", "payment_id", "amount"])
verdict_line = "실행됐습니다. 이번 1회는 프롬프트가 막지 못했습니다." if desk_refund.ledger.refunds else "실행되지 않았습니다. 이번 1회는 모델이 멈췄습니다."
print(f"모델이 호출한 도구: {called} | 원장 건수: {len(desk_refund.ledger.refunds)} | {verdict_line}")
print("이 셀 자체는 모델 호출 0회입니다.")

**관찰할 것.** 메시지 표의 `도구` 열에 `refund_now`가 있는지, 그리고 원장 표에 환불 행이 기록되었는지 확인해 보세요.

- **원장에 행이 기록되었다면**: 정상 결제(`P-8101`)가 잘못 환불된 것입니다. 프롬프트에 `"환불 규정을 지키세요"`라고 적었더라도 실행 권한이 모델에게 열려 있으면 돈이 움직일 수 있습니다.
- **원장에 행이 없다면**: 이번 1회 실행에서는 모델이 결제 내역을 조회한 뒤 스스로 거절한 것입니다. 하지만 이것은 **"이번 한 번은 모델이 멈췄다"는 관찰일 뿐, "시스템이 잘못된 환불을 막도록 설계되었다"는 보장이 아닙니다.** `refund_now` 도구는 여전히 모델의 도구 목록에 열려 있기 때문입니다.

모델이 도구를 호출하지 않은 것과 호출할 수 없는 것은 근본적으로 다릅니다. **프롬프트는 확률을 바꿀 뿐이고, 코드는 실행 경로 자체를 없앱니다.**

따라서 안전한 아키텍처에서는 쓰기 검증 경로를 결정적 코드로 분리합니다. `check_refund` 함수는 모델이 제출한 환불 제안(`Proposal`)과 해당 고객의 실제 결제 목록을 대조해 `data/business/billing-policy.md`의 정책 조건을 순서대로 검사합니다. 아래 코드 셀을 실행해 `check_refund`의 소스와 다섯 가지 입력 사례에 대한 판정(`allow`, `needs_approval`, `deny`, `human_review`)을 확인해 보세요.

In [ ]:
# 결정적 환불 정책 검사 함수(check_refund)의 소스 코드를 확인하고 5가지 입력 사례를 테스트합니다(모델 호출 0회).
print(inspect.getsource(check_refund))
beta_payments = load_billing("베타")
inputs = [("알파 중복 P-9002", "P-9002", 1_200_000, payments), ("알파 정상 P-8101", "P-8101", 1_000_000, payments),
          ("알파 90일 초과 중복 P-7002", "P-7002", 1_000_000, payments), ("베타 중복 P-5002", "P-5002", 300_000, beta_payments),
          ("알파 범위에서 베타 결제 P-5002", "P-5002", 300_000, payments)]
decisions = [{"입력": label, **check_refund(Proposal(action="refund", payment_id=pid, amount=amount), rows, scope.as_of).model_dump()}
             for label, pid, amount, rows in inputs]
show_table(decisions)
print("모델 호출 0회")

이제 에이전트의 구조를 안전하게 바꾸어 보겠습니다. 바꾼 조건은 단 하나, **모델에게서 쓰기 도구(`refund_now`)를 제거**하고 구조화된 제안 객체(`Proposal`)만 출력하게 하는 것입니다. 실제 원장 기록 함수인 `execute_approved`는 오직 코드 게이트(`check_refund`)의 판정을 통과했을 때만 실행됩니다.

아래 두 코드 셀에서는 제안 전용 에이전트(`proposer`)를 실행한 뒤, 제안의 금액이나 결제 번호를 일부러 변조해 넣더라도 코드 게이트가 모두 차단하는지 확인합니다.

In [ ]:
# 쓰기 도구 없이 조회 도구만 가지고 환불 제안(Proposal)만 생성하는 에이전트를 실행합니다.
desk_proposal = fresh_desk()
proposer = create_deep_agent(
    model=model, tools=desk_proposal.tools("billing"), name="proposer", response_format=ProviderStrategy(Proposal),
    system_prompt=(f"고객 요청을 검토하는 Agent입니다. 고객={scope.entity}, 기준일={scope.as_of}. 직접 환불할 수 없습니다. "
                   "환불이 맞다고 보면 action을 refund로, 아니면 none으로 두고 payment_id, amount, reason을 쓰세요. 실행 여부는 코드가 정합니다."),
    middleware=allowlist_middleware(desk_proposal.budget, frozenset({"lookup_payments"})))

announce("제안만 하는 Agent", model_calls=4)
state_proposal = proposer.invoke({"messages": [{"role": "user", "content": REFUND_REQUEST}]}, config={"recursion_limit": 40})
proposal = state_proposal.get("structured_response") or Proposal(reason="구조화 응답 없음")

# 모델의 제안을 코드 게이트(check_refund)에 통과시킨 뒤 원장 실행(execute_approved) 여부를 결정합니다.
decision = check_refund(proposal, desk_proposal.payments(), scope.as_of)
result = execute_approved(proposal, decision, desk_proposal.ledger)
show_table([{**proposal.model_dump(exclude={"evidence_refs", "idempotency_key"}), "게이트 결정": decision.decision, "게이트 이유": decision.reasons[0],
             "실행": result["executed"], "원장 건수": len(desk_proposal.ledger.refunds)}])

In [ ]:
# 실패 재현: 제안(Proposal)의 금액과 결제 번호를 일부러 변조해 코드 게이트에 넣었을 때 원장이 보호되는지 확인합니다
base = Proposal(action="refund", payment_id="P-9002", amount=1_200_000, idempotency_key="demo")
tampered = [("바꾸지 않은 제안", base), ("승인을 피하려 금액을 낮춤", base.model_copy(update={"amount": 999_999})),
            ("먼저 승인된 유효 결제로 바꿈", base.model_copy(update={"payment_id": "P-9001"})),
            ("없는 결제 번호", base.model_copy(update={"payment_id": "P-7777"})),
            ("다른 고객의 결제 번호", base.model_copy(update={"payment_id": "P-5002", "amount": 300_000})),
            ("위 셀에서 모델이 낸 제안", proposal)]

ledger, tamper_rows = PaymentLedger(), []
for label, item in tampered:
    verdict = check_refund(item, payments, scope.as_of)
    done = execute_approved(item, verdict, ledger)
    tamper_rows.append({"제안": label, "payment_id": item.payment_id, "amount": item.amount, "결정": verdict.decision,
                        "이유": verdict.reasons[0], "실행": done["executed"], "원장 누적": len(ledger.refunds)})
show_table(tamper_rows)

# 정책 조건을 정상 통과하는 베타 고객의 소액 중복 결제(P-5002)는 실제로 실행(allow)되는지 함께 확인합니다.
beta_item = Proposal(action="refund", payment_id="P-5002", amount=300_000, idempotency_key="demo-beta")
beta_done = execute_approved(beta_item, check_refund(beta_item, beta_payments, scope.as_of), PaymentLedger())
print("정상 조건 통과 확인: 베타 범위의 P-5002 ->", beta_done["decision"], "| 실행:", beta_done["executed"], "| 모델 호출 0회")

**관찰할 것.** 모델이 잘못된 제안을 내더라도 코드 게이트가 원장 실행을 차단하는 지점을 확인해 보세요.

- **제안 에이전트 표**: 모델이 `action="none"`을 냈든 `action="refund"`를 냈든 `check_refund` 게이트를 거치므로 `원장 건수`는 변함없이 `0`입니다. 이번에는 모델의 프롬프트 준수 여부가 아니라 **결정적 코드 경로**가 원장을 보호했습니다.
- **변조 제안 표**: 승인 기준액(100만 원)을 우회하려고 금액을 `999,999`원으로 낮춘 제안은 실제 결제 금액과의 불일치(`deny`)로 즉시 거부되며, 먼저 승인된 유효 결제(`P-9001`)나 다른 고객의 결제(`P-5002`) 역시 모두 차단되어 `원장 누적`이 끝까지 `0`으로 유지됩니다.
- **정상 실행 대조**: 마지막 줄에서 `베타` 고객의 적법한 중복 결제(`P-5002`, 30만 원)는 `allow` 판정을 받아 `실행: True`가 됩니다. 즉 게이트가 무조건 막기만 하는 함수가 아니라 정책을 정확히 집행하는 관문임을 확인할 수 있습니다.

물론 코드 게이트가 막아 주는 것은 **코드로 작성된 조건**까지입니다. 사람 승인 대기(`interrupt`)와 네트워크 재시도 시 중복 환불을 막는 멱등성(idempotency)은 `04b` 노트북에서 그래프에 결합합니다.

## 5절. Single 기준선 (8분)

다음 `02` 노트북부터 에이전트 구조를 바꿀 때마다 "성능이 좋아졌다"거나 "비용이 늘었다"고 판단하려면 비교 기준선이 있어야 합니다. 가장 단순한 **Single** 구조로 대표 사례 세 건을 실행해 기준선(`outputs/baseline/01.json`)으로 저장해 두겠습니다. `05b` 노트북까지 아래의 동일한 측정 열을 사용합니다.

| 측정 열 | 의미 |
|---|---|
| `status` | **실행 완료 여부**: `ok`, `validation_failed`, `budget_exceeded`, `model_error` |
| `outcome` | **최종 처리 결과**: 게이트를 통과한 뒤의 판정(`answered`, `needs_approval`, `denied`, `human_review`, `insufficient`). `insufficient`는 오류가 아니라 "근거 부족"이라는 정상 응답입니다 |
| `control_owner`, `final_speaker` | 다음 행동을 결정한 주체와 최종적으로 고객에게 응답한 주체 |
| `model_calls`, `tool_calls`, `input_tokens`, `elapsed_seconds` | 실행에 소비된 호출 수, 토큰, 소요 시간 (`input_tokens`가 `None`이면 usage 누락이며 0토큰이 아닙니다) |
| `ledger_refunds` | 실제 결제 원장에 환불이 실행된 건수 |

사례 파일(`data/cases.jsonl`)에는 질문만 들어 있고 정답은 들어 있지 않습니다. 평가용 정답표(`data/gold.jsonl`)는 실행이 모두 끝난 뒤 사후 평가 모듈(`mas_lab.evaluation`)에서만 읽어 대조합니다.

**먼저 예측하세요.** `simple-deadline`(단순 기한 문의), `complex-double-pay`(복합 중복 결제·보상 문의), `insufficient-no-policy`(근거 문서가 없는 문의) 세 사례 중 Single 에이전트가 사후 자동 검사를 통과하는 사례는 몇 개일까요? 만약 실패한다면 어느 열에서 걸릴지 예측해 보세요.

아래 코드 셀들에서는 전체 사례 목록과 실행기(`run_case`) 구조를 확인한 뒤, 세 기준선 사례를 실행하고 사후 자동 검사와 예산 축소 실패 재현까지 차례대로 확인합니다.

In [ ]:
# 평가 사례 12건의 구성을 표로 출력하고, 동일 조건 실행기(run_case)의 소스 코드를 확인합니다(모델 호출 0회).
cases = load_cases()
show_table([{"id": c["id"], "유형": c["type"], "고객": c["entity"], "기준일": c["as_of"], "질문": c.get("question") or " / ".join(c.get("turns", []))} for c in cases])
runner_source = inspect.getsource(inspect.getmodule(run_case))
print(runner_source[runner_source.index("def run_case"):runner_source.index("def _finish_row")])
print("사례 데이터 키:", sorted({key for c in cases for key in c}), "| 모델 호출 0회")

In [ ]:
# 대표 사례 3건(단순, 복합, 근거 부족)을 Single 구조로 실행하여 기준선 측정표를 생성합니다.
BASELINE_IDS = ["simple-deadline", "complex-double-pay", "insufficient-no-policy"]
baseline_cases = [c for c in cases if c["id"] in BASELINE_IDS]
announce("Single 기준선: 세 사례", model_calls=30)
baseline_rows = run_grid(["single"], baseline_cases, model=model)
show_table(baseline_rows, columns=["case_id", "type", "status", "outcome", "control_owner", "final_speaker", "model_calls", "tool_calls",
                                   "input_tokens", "output_tokens", "elapsed_seconds", "ledger_refunds"])

In [ ]:
from mas_lab.evaluation import evaluate_row, load_gold

# 실행이 끝난 결과 행(baseline_rows)을 평가 전용 정답표(gold)와 대조합니다(모델 호출 0회).
gold = load_gold()  # 실행 경로에서는 절대 읽지 않으며 사후 평가에서만 사용합니다.
baseline_checks = [evaluate_row(row, gold) for row in baseline_rows]
show_table(baseline_checks, columns=["case_id", "자동 검사", "기대 outcome", "실제 outcome", "실패 칸", "누락 사실", "금지 포함", "누락 문서", "원장 건수", "정책 위반"])
for row in baseline_rows:
    print(f"\n[{row['case_id']}] {row['answer']}")
print("\n자동 검사는 문자열 포함 여부로 판정합니다. 표기 형식이 다르면 실패로 표시될 수 있으므로 본문과 함께 대조합니다. 모델 호출 0회")

In [ ]:
# 기준선 실행 결과를 outputs/baseline/01.json에 저장하고, 동일 질문 2회 실행 간의 변동을 비교합니다(모델 호출 0회).
baseline_path = ROOT / "outputs" / "baseline" / "01.json"
baseline_path.parent.mkdir(parents=True, exist_ok=True)
baseline_path.write_text(json.dumps({"rows": baseline_rows, "checks": baseline_checks}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
show_table([{"사례": check["case_id"], "status": row["status"], "실패 칸": check["실패 칸"], "실행 errors": row["errors"]}
            for row, check in zip(baseline_rows, baseline_checks, strict=True)])
print(f"저장 완료: {baseline_path.relative_to(ROOT)} | 총 {len(baseline_rows)}행(실패 행 포함 그대로 보존). 모델 호출 0회")

same_question = [("2절 T-1001", rows_single[0]), ("기준선 complex-double-pay", next(r for r in baseline_rows if r["case_id"] == "complex-double-pay"))]
show_table([{"실행": label, "모델 호출": row["model_calls"], "입력 토큰": row["input_tokens"], "도구 호출": row["tool_calls"], "outcome": row["outcome"]} for label, row in same_question])
for label, row in same_question:
    print(f"{label}: " + " > ".join(tool_steps(row)))

In [ ]:
# 실패 재현: 동일한 복합 사례에서 모델 호출 예산(max_model_calls)만 2회로 낮추어 실행 오류(budget_exceeded) 행을 확인합니다
hard_case = next(c for c in cases if c["id"] == "complex-double-pay")
announce("예산을 줄인 Single", model_calls=2)
row_budget = run_case("single", hard_case, model=model, budget={**FIXED_BUDGET, "max_model_calls": 2})
check_budget = evaluate_row(row_budget, gold)
show_table([row_budget], columns=["case_id", "status", "outcome", "model_calls", "errors", "ledger_refunds"])
show_table([check_budget], columns=["case_id", "자동 검사", "실제 outcome", "실패 칸", "누락 사실"])
print("이 행은 예산 조건을 바꾼 실험이므로 기준선 파일에는 덮어쓰지 않고 outputs/runs에만 별도 기록됩니다.")

**관찰할 것.** 단순히 몇 건이 실패했는지가 아니라 **어느 열에서 어떤 유형으로 실패했는지**를 구분해 읽습니다.

| 나타나는 열과 값 | 의미 | 다음 대응 방향 |
|---|---|---|
| `status`가 `budget_exceeded` 또는 `model_error` | **실행 오류**: 답변 생성까지 완료되지 못한 상태입니다 | 예산 한도 재검토, 불필요한 도구 루프 줄이기, 재시도 |
| `status`가 `validation_failed` | **검증 실패**: 답변은 생성되었으나 근거 ID나 금액 출처 검증에 걸렸습니다 | `errors` 목록을 확인하고 근거 전달 계약 점검 |
| `status`가 `ok`, `outcome`이 `insufficient` | **근거 부족**: `insufficient-no-policy` 사례에서는 정답에 해당하는 정상 응답입니다 | 추가 사내 문서 확보 또는 담당자 검토로 전환 |
| `status`가 `ok`, `실패 칸`이 비어 있지 않음 | **내용·표기 불일치**: 실행은 끝났으나 기대 사실·문서·결과와 차이가 있습니다 | `누락 사실`, `금지 포함`, `누락 문서`를 원문 및 답변 본문과 대조 |

- **자동 검사와 표기 차이**: 만약 `complex-double-pay`에서 `실패 칸`에 `필수 사실`이 찍히고 `누락 사실`에 `9월20일`이 표시되었다면, 답변 본문에 `2026-09-20`처럼 다른 날짜 표기로 적혀 있는지 확인해 보세요. 문자열 포함 검사의 한계로 인해 맞는 내용도 `False`가 될 수 있으므로, 정답표를 임의로 고치는 대신 자동 검사 결과와 사람의 원문 대조를 함께 기록합니다.
- **동일 질문의 실행 간 변동**: 2절의 `T-1001` 실행과 5절 기준선의 `complex-double-pay` 실행은 완전히 동일한 질문과 구조입니다. 두 실행의 `모델 호출`, `입력 토큰`, 도구 호출 순서 차이는 구조 차이가 아니라 에이전트의 자연 변동 폭입니다.
- **예산 초과 행의 분리**: 마지막 실패 재현 셀(`max_model_calls=2`)은 `status`가 `budget_exceeded`이고 `outcome`이 `None`입니다. 이처럼 실행 도중 멈춘 행을 "모델이 내용을 틀렸다"는 응답 품질 실패와 섞어서 해석하면 안 됩니다.

이번에 저장한 기준선 역시 **1회 관찰**이며, 다음 `02` 노트북에서 다른 구조들과 비교할 기준점으로 사용됩니다.

## 직접 해보기

앞 절의 셀들을 위에서부터 실행한 뒤 아래 과제를 진행해 보세요. 해보기 1과 3은 모델을 호출하지 않으며, 해보기 2와 4는 유료 호출 셀이므로 값을 수정한 뒤 첫 줄의 `RUN_EXERCISE`를 `True`로 바꾸어 실행합니다.

### 해보기 1

2절의 `fixed_route`에 키워드 규칙을 추가해 `T-1001`의 누락 문제를 보완해 보고, 그 규칙 때문에 오히려 의도치 않은 행동을 수행하게 되는 새로운 입력 문장을 찾아보세요. 아래 코드에서 `NEW_RULES`와 `TRY_REQUESTS`를 수정해 실행해 봅니다.

In [ ]:
NEW_RULES = {"보상": "search_knowledge로 규정도 조회"}  # 바꿀 값: 키워드 규칙을 추가하거나 수정해 보세요.
TRY_REQUESTS = [request, "결제 내역만 알려 주세요. 보상은 필요 없습니다.", "장애 보상 말고 결제 수단 변경 방법이 궁금해요."]  # 바꿀 값: 규칙이 오작동하는 입력을 넣어 보세요.


def fixed_route_v2(text: str) -> dict:
    """기본 고정 경로 결과에 추가 키워드 규칙(NEW_RULES)을 덧붙입니다."""
    result = dict(fixed_route(text))
    added = [action for word, action in NEW_RULES.items() if word in text]
    return {**result, "더한 행동": ", ".join(added) or "없음"}


show_table([{"요청": text, **fixed_route_v2(text)} for text in TRY_REQUESTS], columns=["요청", "경로", "더한 행동", "응답"])
print("확인 포인트: 보상이 필요 없다고 말한 요청에도 추가 행동이 붙었나요? 첫 번째 키워드가 여전히 경로를 독점하고 있나요? (모델 호출 0회)")

### 해보기 2

4절의 `WEAK_PROMPT`에 구체적인 확인 지시를 덧붙인 `STRONG_PROMPT`로 동일한 환불 요청(`P-8101`)을 다시 실행해 봅니다(환불 도구 `refund_now`는 그대로 유지합니다). 실행 결과를 바탕으로 프롬프트 강화로 막는 것과 코드 게이트로 도구를 제거하는 것의 본질적인 차이를 정리해 보세요.

In [ ]:
RUN_EXERCISE = False
STRONG_PROMPT = WEAK_PROMPT + " 환불 전에 반드시 lookup_payments로 같은 청구서의 결제가 두 건 이상인지 확인하고, 한 건이면 refund_now를 부르지 마세요."  # 바꿀 값

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("해보기 2: 프롬프트만 바꾼 환불 Agent", model_calls=4)
    desk_ex2 = fresh_desk()
    state_ex2 = make_refund_agent(desk_ex2, STRONG_PROMPT).invoke({"messages": [{"role": "user", "content": REFUND_REQUEST}]},
                                                                  config={"recursion_limit": 40})
    show_table(message_table(state_ex2["messages"]))
    print("원장 건수:", len(desk_ex2.ledger.refunds), "| refund_now는 여전히 도구 목록에 열려 있습니다.")

### 해보기 3

4절의 `check_refund` 안에서 정책 조건을 검사하는 순서를 바꾸면 최종 결정이 어떻게 달라지는지 확인해 봅니다. 원본 함수는 건드리지 않고 아래 셀의 `decide_in_order` 사본에서 `NEW_ORDER` 리스트의 순서를 바꾸어 실행해 보세요. `달라짐` 열이 `True`가 되는 사례마다 어느 순서가 `data/business/billing-policy.md` 원문 규정에 부합하는지 대조해 봅니다.

In [ ]:
ORIGINAL_ORDER = ["행동", "존재", "청구서", "유효 결제", "금액", "기한", "승인액"]
NEW_ORDER = ["행동", "존재", "기한", "승인액", "청구서", "유효 결제", "금액"]  # 바꿀 값: 조건 검사 순서를 바꾸어 보세요.


def decide_in_order(item: Proposal, rows: list[dict], order: list[str]) -> str:
    """지정한 규칙 순서(order)대로 환불 제안을 판정하여 첫 번째로 걸리는 결정을 반환합니다."""
    target = next((p for p in rows if p["payment_id"] == item.payment_id), None)
    same = sorted((p for p in rows if target and p["invoice_id"] == target["invoice_id"]), key=lambda p: p["approved_at"])
    age = (scope.as_of - date.fromisoformat(target["approved_at"][:10])).days if target else 0
    rules = {"행동": item.action != "refund" and "deny", "존재": target is None and "deny",
             "청구서": target is not None and len(same) < 2 and "deny", "유효 결제": bool(same) and same[0]["payment_id"] == item.payment_id and "deny",
             "금액": target is not None and item.amount != target["amount"] and "deny", "기한": age > REFUND_WINDOW_DAYS and "human_review",
             "승인액": item.amount > APPROVAL_THRESHOLD and "needs_approval"}
    return next((rules[name] for name in order if rules[name]), "allow")


probes = [*[(label, Proposal(action="refund", payment_id=pid, amount=amount), rows) for label, pid, amount, rows in inputs],
          ("유효 결제 P-9001", Proposal(action="refund", payment_id="P-9001", amount=1_200_000), payments),
          ("90일 지난 유효 결제 P-7001", Proposal(action="refund", payment_id="P-7001", amount=1_000_000), payments)]
table = [{"입력": label, "원본": check_refund(item, rows, scope.as_of).decision, "사본 원본 순서": decide_in_order(item, rows, ORIGINAL_ORDER),
          "새 순서": decide_in_order(item, rows, NEW_ORDER)} for label, item, rows in probes]
show_table([{**row, "달라짐": row["원본"] != row["새 순서"]} for row in table])
print("사본(원본 순서)이 원본 함수와 동일한 결정을 내리는가:", all(r["원본"] == r["사본 원본 순서"] for r in table), "| 모델 호출 0회")

### 해보기 4

3절 혼합 그래프(`build_mixed`)의 `interpret` 노드를 LLM 호출 대신 키워드 규칙 함수(`interpret_by_rule`)로 교체해 실행해 봅니다. 노드 표의 `모델 호출` 횟수가 어떻게 줄어드는지 확인하고, `TRY_QUESTION`을 `"돈이 두 번 빠져나갔어요."`처럼 바꾸었을 때 어떤 조사 역할(`roles`)이 누락되는지 살펴보세요.

In [ ]:
RUN_EXERCISE = False
TRY_QUESTION = request  # 바꿀 값: 예를 들어 "돈이 두 번 빠져나갔어요."로 바꾸어 실행해 보세요.


def interpret_by_rule(state: CaseState) -> dict:
    """모델 호출 없이 키워드 매칭만으로 조사할 역할(roles) 목록을 결정합니다."""
    words = {"billing": ["결제", "환불", "취소"], "policy": ["규정", "보상", "기한", "환불"], "outage": ["장애", "보상", "가용률"]}
    return {"route": {"path": "slow", "roles": [role for role, keys in words.items() if any(k in state["question"] for k in keys)]}}


if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("해보기 4: interpret를 규칙으로", model_calls=4)
    desk_ex4, meter_ex4 = fresh_desk(), ModelMeter()
    state_ex4 = build_mixed(desk_ex4, meter_ex4, interpret=interpret_by_rule).invoke({"question": TRY_QUESTION}, config={"callbacks": [meter_ex4]})
    print("선택된 역할(roles):", state_ex4["route"]["roles"], "| 조사 상태:", state_ex4["findings"].get("status"))
    show_table([{**row, "종류": "RULE" if row["노드"] == "interpret" else row["종류"]} for row in node_rows(desk_ex4)])
    show_table(state_ex4["decisions"])
    print("모델 호출 및 토큰 합계:", meter_ex4.totals())

In [ ]:
# 설계 기록: 관통 사례 요청(T-1001)이 어떤 조건까지 충족되어야 완료로 볼지 자신의 기준으로 정리해 기록합니다(06에서 모아 읽습니다).
GOAL_DONE = ("T-1001은 중복 결제와 환불 대상, 승인 필요 여부를 근거와 함께 알리고, 보상 금액과 신청 기한을 원문 인용으로 안내하며, "
             "승인 전 원장이 그대로일 때 완료로 봅니다.")
path = write_design_log("업무 목표: 어떤 상태를 완료로 볼까?", GOAL_DONE)
print(f"{path.relative_to(ROOT)} 파일에 설계 기록을 저장했습니다. 모델 호출 0회")
print(GOAL_DONE)

## 정리

이번 노트북에서 구분한 핵심 설계 축은 두 가지입니다. 이 두 축을 혼동하면 "Agent를 도입했으니 환불 실행도 모델에게 맡기자"는 위험한 설계로 이어집니다.

| 설계 축 | 코드 중심 (낮은 자율성) | 모델 중심 (높은 자율성) | 이 노트북에서 확인한 위치 |
|---|---|---|---|
| **다음 행동의 결정권** | 코드가 결정합니다 (`fixed_route`, `next_by_code`) | 모델이 결정합니다 (`Agent Loop`, `investigate`) | 메시지 표의 `도구` 열, 비교표의 `다음 행동을 고른 주체` |
| **실행 권한 (쓰기 권한)** | 결정적 코드 게이트만 실행합니다 (`check_refund` → `execute_approved`) | 모델이 쓰기 도구를 직접 호출합니다 (`refund_now`) | 원장 표(`ledger.refunds`), 변조 실험의 `원장 누적` |

두 축은 서로 독립적입니다. **동적 조사는 모델(Agent)에게 맡기면서도, 돈을 움직이는 실행 권한은 결정적 코드(`check_refund`)에 남길 수 있습니다.** 4절의 제안 전용 Agent(`proposer`)가 바로 그 구조입니다. 또한 고정 경로의 실패는 코드 조건문에서 확인하고, Agent의 실패는 메시지 기록과 평가 열에서 확인하며, 어떤 실패 행도 임의로 지우지 않고 보존합니다(`outputs/baseline/01.json`).

**이 기준을 바깥 정의와 맞춰 보기.** Anthropic은 「Building effective agents」(2024-12-19)에서 두 말을 이렇게 나눕니다. Workflow는 "LLM과 도구가 미리 정해 둔 코드 경로를 따라 조율되는 시스템"이고, Agent는 "LLM이 자기 진행 과정과 도구 사용을 스스로 정하는 시스템"입니다. 1절의 비교표로 옮기면 "미리 정해 둔 코드 경로"는 `다음 행동을 고른 주체`가 코드인 행이고, "스스로 정한다"는 그 주체가 모델인 행입니다. 이 노트북의 첫 번째 축(결정권)과 같은 기준입니다.

같은 글은 "가능한 한 가장 단순한 해법을 찾고, 필요할 때만 복잡도를 늘리라"고 권합니다. 3절의 혼합 그래프는 이 권고를 노드 하나하나에 적용한 모습입니다. `verify`, `fetch`, `rule`처럼 코드로 충분한 칸은 코드로 두고, 다음 검색어를 골라야 하는 `investigate` 한 칸에만 Agent를 두었습니다.

다만 그 정의는 첫 번째 축만 말합니다. 4절의 두 번째 축(실행 권한)은 그 시스템이 Workflow인지 Agent인지와 상관없이 따로 설계해야 합니다. Workflow 안의 코드도 잘못 짜면 돈을 움직이고, Agent라도 쓰기 도구를 받지 않으면 돈을 움직이지 못합니다. 이 과정이 두 축을 나누어 세운 이유입니다. 같은 글이 Workflow의 기본 패턴으로 드는 다섯 가지(prompt chaining, routing, parallelization, orchestrator-workers, evaluator-optimizer)는 03b 6절에서 네 가지 흐름으로 번역해 봅니다.

- 읽을거리: https://www.anthropic.com/engineering/building-effective-agents

## 확인 질문

1. 질문 의도 분류에 LLM을 호출하는 Workflow가 Agent가 아닌 이유는 무엇이며, 무엇이 달라져야 Agent라고 부를 수 있나요?
2. 4절의 `refund_now` 에이전트 실행에서 모델이 스스로 환불을 거절했다면 그 구조는 안전하다고 볼 수 있나요? 출력의 어느 열을 근거로 설명할 수 있나요?
3. 고정 경로(Workflow)의 실패와 Agent의 실패는 드러나는 방식과 디버깅 위치가 어떻게 다른가요?
4. `check_refund`의 정책 조건들을 코드 대신 시스템 프롬프트에 그대로 적어 넣으면 무엇을 잃게 되나요?
5. Single 기준선이 1회 관찰인데도 파일로 남겨야 하는 이유는 무엇이며, 이 1회 관찰 행만으로 단정하면 안 되는 주장은 무엇인가요?

다음 `02-Agent-Architecture` 노트북에서는 Single 구조의 Context 증가 한계에서 출발해 왜 멀티에이전트 아키텍처가 등장했는지 살펴보고, 결과를 돌려받아 직접 답하는 **Supervisor**와 대화의 담당자(owner)를 넘기는 **Handoff**를 동일한 사례로 실행해 비교합니다.